In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
import numpy as np
import pickle

In [2]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
#preprocessing 
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  str    
 2   Gender           10000 non-null  str    
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), str(2)
memory usage: 966.1 KB


In [5]:
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])

onehot_encoder = OneHotEncoder()
country_encoded = onehot_encoder.fit_transform(data[['Geography']]).toarray()
country_encoded_df = pd.DataFrame(country_encoded, columns=onehot_encoder.get_feature_names_out(['Geography']))
country_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [6]:
input_data = pd.concat([data.drop('Geography', axis=1), country_encoded_df], axis=1)
input_data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [7]:
#Splitting the dataset into features and target variable
X = input_data.drop('EstimatedSalary', axis=1)
y = input_data['EstimatedSalary']

In [8]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [11]:
with open('scaling.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

with open('onehot_encoder.pkl', 'wb') as f:
    pickle.dump(onehot_encoder, f)


In [12]:
##ANN

model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(x_train_scaled.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

import tensorboard
import datetime
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir='./logs/fit' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S"), histogram_freq=1)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


model.fit(x_train_scaled, y_train, validation_data=(x_test_scaled, y_test), epochs=100, batch_size=32, validation_split=0.2, callbacks=[tensorboard_callback, early_stopping_callback])

c:\Users\DELL\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 13381088256.0000 - mae: 100365.2969 - val_loss: 12997495808.0000 - val_mae: 98469.9688
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 13199251456.0000 - mae: 99463.1719 - val_loss: 12640216064.0000 - val_mae: 96665.4141
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 12566415360.0000 - mae: 96322.8047 - val_loss: 11736237056.0000 - val_mae: 92099.5859
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 11332961280.0000 - mae: 90221.8438 - val_loss: 10250939392.0000 - val_mae: 84614.9922
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9583300608.0000 - mae: 81512.0234 - val_loss: 8388891136.0000 - val_mae: 75291.8516
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7622379520.0000 - mae: 71610.4844 - val_loss: 6527936512.0000 - val_mae: 66060.9766
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 5855478784.0000 - mae: 62630.8008 - val_loss: 5048908800.000

In [13]:
%load_ext tensorboard

In [14]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 8684), started 6:52:41 ago. (Use '!kill 8684' to kill it.)

In [15]:
#evaluate the model
loss, mae = model.evaluate(x_test_scaled, y_test)
print(f"Test Loss: {loss}, Test MAE: {mae}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3356440832.0000 - mae: 50117.4219
Test Loss: 3356440832.0, Test MAE: 50117.421875


In [16]:
model.save('salary_regression_model.h5')